In [14]:
# imports
import io
import re
from pathlib import Path
import json
import pandas as pd

In [5]:
# folder with yearly files: 2014.csv ... 2024.csv
DATA_DIR = Path("../Datasets/Rankings")

In [3]:
# load one tab-separated file, fix World Rank split across two lines (2017/2023/2024)
# standardize country column name, coerce Score to numeric, tag with Year
def load_year_file(path: Path) -> pd.DataFrame:
    raw_text = path.read_text(encoding="utf-8", errors="replace")

    wrapped_rank_pattern = re.compile(r"(\d+)\s*\nTop\s+[\d.]+%\t")
    clean_text = wrapped_rank_pattern.sub(r"\1\t", raw_text)

    df = pd.read_csv(io.StringIO(clean_text), sep="\t")

    df = df.rename(columns={"Country": "Location"})

    df["World Rank"] = df["World Rank"].astype(str).str.extract(r"(\d+)").astype("Int64")
    df["Score"] = pd.to_numeric(df["Score"], errors="coerce")

    year = int(re.search(r"(\d{4})", path.stem).group(1))
    df["Year"] = year

    return df

In [6]:
# load all years, keep columns common to every year, concatenate
def build_consistent_dataframe(data_dir: Path) -> pd.DataFrame:
    files = sorted(data_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {data_dir!r}")

    per_year_dfs = [load_year_file(f) for f in files]

    common_cols = set(per_year_dfs[0].columns)
    for df in per_year_dfs[1:]:
        common_cols &= set(df.columns)
    common_cols.discard("Year")

    ordered_common_cols = [c for c in per_year_dfs[0].columns if c in common_cols]
    final_cols = ordered_common_cols + ["Year"]

    combined = pd.concat([df[final_cols] for df in per_year_dfs], ignore_index=True)
    combined = combined.rename(columns={"Location": "Country"})

    return combined


df = build_consistent_dataframe(DATA_DIR)
df.head(10)

,World Rank,Institution,Country,National Rank,Education Rank,Employability Rank,Faculty Rank,Score,Year
0,1,Harvard University,USA,1,1,1,1,100.00,2014
1,2,Stanford University,USA,2,11,2,4,99.09,2014
2,3,Massachusetts Institute of Technology,USA,3,3,11,2,98.69,2014
3,4,University of Cambridge,United Kingdom,1,2,10,5,97.64,2014
4,5,University of Oxford,United Kingdom,2,7,12,10,97.51,2014
5,6,Columbia University,USA,4,13,8,9,97.41,2014
6,7,"University of California, Berkeley",USA,5,4,22,6,92.84,2014
7,8,University of Chicago,USA,6,10,14,8,92.03,2014
8,9,Princeton University,USA,7,5,16,3,88.56,2014
9,10,Yale University,USA,8,9,25,11,88.11,2014


In [8]:
# keep European (+ Turkey) institutions only
europe_list = [
    "Albania", "Andorra", "Austria", "Belgium", "Bosnia and Herzegovina", "Bulgaria",
    "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland", "France", "Germany",
    "Greece", "Hungary", "Iceland", "Ireland", "Italy", "Kosovo", "Latvia", "Liechtenstein",
    "Lithuania", "Luxembourg", "Malta", "Moldova", "Monaco", "Montenegro", "Netherlands",
    "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "San Marino", "Serbia",
    "Slovakia", "Slovenia", "Spain", "Sweden", "Switzerland", "Turkey", "Ukraine", "United Kingdom",
    "Vatican"
]

df_europe = df[df["Country"].isin(europe_list)].reset_index(drop=True)
df_europe.head(10)

,World Rank,Institution,Country,National Rank,Education Rank,Employability Rank,Faculty Rank,Score,Year
0,4,University of Cambridge,United Kingdom,1,2,10,5,97.64,2014
1,5,University of Oxford,United Kingdom,2,7,12,10,97.51,2014
2,18,Swiss Federal Institute of Technology in Zurich,Switzerland,1,16,105,13,72.18,2014
3,30,University College London,United Kingdom,3,20,406,52,61.05,2014
4,35,École normale supérieure - Paris,France,1,8,478+,59,59.72,2014
5,36,École Polytechnique,France,2,150,6,208,59.54,2014
6,39,Imperial College London,United Kingdom,4,121,98,38,58.85,2014
7,50,University of Paris-Sud,France,3,26,410,25,56.06,2014
8,56,University of Edinburgh,United Kingdom,5,42,131,36,55.20,2014
9,68,Pierre-and-Marie-Curie University,France,4,36,431,86,53.91,2014


In [9]:
# rank institutions within Europe only, per year, by Score
df_europe["European Rank"] = (
    df_europe.groupby("Year")["Score"]
    .rank(ascending=False, method="min")
    .astype("Int64")
)

In [10]:
# final output: institution, country, Year, European Rank only
df_final = (
    df_europe[["Institution", "Country", "Year", "European Rank"]]
    .sort_values(["Year", "European Rank"])
    .reset_index(drop=True)
)
df_final.head(10)

,Institution,Country,Year,European Rank
0,University of Cambridge,United Kingdom,2014,1
1,University of Oxford,United Kingdom,2014,2
2,Swiss Federal Institute of Technology in Zurich,Switzerland,2014,3
3,University College London,United Kingdom,2014,4
4,École normale supérieure - Paris,France,2014,5
5,École Polytechnique,France,2014,6
6,Imperial College London,United Kingdom,2014,7
7,University of Paris-Sud,France,2014,8
8,University of Edinburgh,United Kingdom,2014,9
9,Pierre-and-Marie-Curie University,France,2014,10


In [11]:
df_final.to_csv("../Datasets/Processed/European_rankings.csv", index=False)

# Merging the datasets

In [ ]:
mobility = pd.read_csv("../Datasets/Processed/df_mobility_hicp.csv", index_col=0)
rankings = pd.read_csv("../Datasets/Processed/european_rankings.csv")
name_map = json.load(open("../university_flat_mapping.json", 'r', encoding='utf-8'))

C:\Users\rpasq\AppData\Local\Temp\ipykernel_20472\3487683994.py:1: DtypeWarning: Columns (0: Participant Age) have mixed types. Specify dtype option on import or set low_memory=False.
  mobility = pd.read_csv("../Datasets/Processed/df_mobility_hicp.csv", index_col=0)


Country aliases not covered by the ranking dataset.

In [26]:
COUNTRY_ALIASES = {"czechia": "czech republic", "slovakia": "slovak republic", "türkiye": "turkey", "turkiye": "turkey"}

def norm_country(s):
    s = s.str.strip().str.casefold()
    return s.map(lambda c: COUNTRY_ALIASES.get(c, c))

def norm_institution(s):
    return s.map(name_map).fillna(s).str.strip().str.casefold()

In [28]:
rank_key = rankings.assign(
    inst_key=norm_institution(rankings["Institution"]).astype("category"),
    country_key=norm_country(rankings["Country"]).astype("category"),
)[["inst_key", "country_key", "Year", "European Rank"]]

In [27]:
mobility["send_inst_key"] = norm_institution(mobility["Sending Organization"]).astype("category")
mobility["send_country_key"] = norm_country(mobility["Sending Country"]).astype("category")
mobility["recv_inst_key"] = norm_institution(mobility["Receiving Organization"]).astype("category")
mobility["recv_country_key"] = norm_country(mobility["Receiving Country"]).astype("category")

Join on institution, country and year.

In [32]:
merged = mobility.merge(
    rank_key.rename(columns={
        "inst_key": "send_inst_key", "country_key": "send_country_key",
        "Year": "Academic Year", "European Rank": "Sending Institution Rank",
    }),
    on=["send_inst_key", "send_country_key", "Academic Year"], how="left",
)

merged = merged.merge(
    rank_key.rename(columns={
        "inst_key": "recv_inst_key", "country_key": "recv_country_key",
        "Year": "Academic Year", "European Rank": "Receiving Institution Rank",
    }),
    on=["recv_inst_key", "recv_country_key", "Academic Year"], how="left",
)

merged = merged.drop(columns=["send_inst_key", "send_country_key", "recv_inst_key", "recv_country_key"])
merged.to_csv("../Datasets/Processed/hicp_mobility_ranking_dataset.csv", index=False)
merged.head(10)

,Mobility Duration,Field of Education,Academic Year,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending Country HICP,Sending City,Sending Organization,Receiving Country,Receiving Country HICP,Receiving City,Receiving Organization,Sending Institution Rank,Receiving Institution Rank
0,61,Business and administration,2023,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,Austria,130.4,WIEN,WU,Germany,125.90,Munich,-,NaN,NaN
1,101,Economics,2023,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,0,27,Austria,130.4,WIEN,WU,France,120.50,Toulouse,Université Toulouse Capitole,NaN,NaN
2,121,Business and administration,2023,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,Austria,130.4,WIEN,WU,France,120.50,Paris,-,NaN,NaN
3,171,Business and administration,2023,Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,22,Austria,130.4,WIEN,WU,France,120.50,Paris,-,NaN,NaN
4,86,Business and administration,2023,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,23,Austria,130.4,WIEN,WU,Germany,125.90,Hamburg,-,NaN,NaN
5,60,Business and administration,2023,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,22,Austria,130.4,WIEN,WU,France,120.50,Paris,-,NaN,NaN
6,91,Business and administration,2023,Serbia,ISCED-7 - Second cycle / Master’s or equivalen...,Female,0,23,Austria,130.4,WIEN,WU,Portugal,118.98,Lisbon,-,NaN,NaN
7,66,Economics,2023,Italy,ISCED-7 - Second cycle / Master’s or equivalen...,Female,0,25,Austria,130.4,WIEN,WU,Italy,120.90,Bozen,-,NaN,NaN
8,88,Business and administration,2023,Austria,ISCED-7 - Second cycle / Master’s or equivalen...,Male,0,24,Austria,130.4,WIEN,WU,Netherlands,127.81,Amsterdam,-,NaN,NaN
9,90,Business and administration,2023,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,24,Austria,130.4,WIEN,WU,Sweden,126.44,Stockholm,-,NaN,NaN
